# LaStBeRu watchlist watchlist × DP2

Este notebook genera un análisis completo, fuera del dominio temporal, para `LaStBeRu_LaStBeRu watchlist_watchlist.csv`. El catálogo confirmado está contenido en él (511 filas con coincidencia exacta de nombre y coordenadas) y no se analiza por separado.

Se guardan: cobertura de visitas CCD individuales por banda, disponibilidad de coadds, profundidad y PSF, tablas por catálogo y `grade`, distribuciones, mapas del cielo coloreados por cantidad de visitas, mosaicos de coadds de 20″ × 20″ y FITS por banda.

La cobertura y los coadds se consultan para `watchlist` y se reutilizan mediante caches; todos los outputs quedan bajo la carpeta `watchlist`.

## 1. Configuración e imports

Esta celda define DP2, las colecciones, las bandas, las carpetas de salida y las caches. Los FITS se guardan fuera de la carpeta de resultados, en `delete_sundays/lastberu_cutouts/watchlist/fits`.

In [1]:
from pathlib import Path
import hashlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from target_selection import build_collection_report_pdf, open_collection

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data_release_config.py').exists()), Path.cwd())
DATA_DIR = REPO_ROOT / 'data' / 'LaStBeRu_data'
CATALOG_FILES = {'watchlist': DATA_DIR / 'LaStBeRu_watchlist_LSST_grades.csv'}
BANDS = tuple('ugrizy')
COLLECTION = open_collection('DP2')
OUTPUT_ROOT = REPO_ROOT / 'notebooks' / 'outputs' / 'lastberu_dp2'
CACHE_ROOT = OUTPUT_ROOT / 'cache'
FITS_ROOT = Path('/deleted-sundays/nowokaren/lastberu_dp2_cutouts/fits')
PNG_ROOT = Path('/deleted-sundays/nowokaren/lastberu_dp2_cutouts/pngs')
CUTOUT_SIZE_ARCSEC = 20.0
SEARCH_RADIUS_DEG = 0.35
TILE_NSIDE = 64
TAP_WORKERS = 2
MAKE_CUTOUTS = True
for path in (OUTPUT_ROOT, CACHE_ROOT, FITS_ROOT, PNG_ROOT): path.mkdir(parents=True, exist_ok=True)


## 2. Carga y normalización de los catálogos

Se lee el CSV `watchlist`, se validan RA/DEC y se agrega `catalog` para conservar el origen. `target_id` incluye el nombre del catálogo.

In [2]:
frames = []
for catalog_name, path in CATALOG_FILES.items():
    if not path.exists(): raise FileNotFoundError(path)
    frame = pd.read_csv(path).copy()
    required = {"object_id", "ra", "dec", "extra_info"}
    missing = required - set(frame.columns)
    if missing: raise ValueError(f"{path} does not contain {sorted(missing)}")
    frame["RA"] = pd.to_numeric(frame["ra"], errors="coerce")
    frame["DEC"] = pd.to_numeric(frame["dec"], errors="coerce")
    frame = frame.dropna(subset=["object_id", "RA", "DEC"]).reset_index(drop=True)
    frame["DB_JNAME"] = frame["object_id"].astype(str)
    frame["catalog"] = catalog_name
    def first_extra_value(value):
        text = "" if pd.isna(value) else str(value).strip()
        first = text.split(",", 1)[0]
        value = first.split("=", 1)[1].strip() if "=" in first else first
        return value if value and value.lower() not in {"nan", "none"} else "unavailable"
    frame["grade"] = frame["extra_info"].map(first_extra_value)
    frame["catalog_index"] = np.arange(len(frame))
    frame["target_id"] = catalog_name + ":" + frame["object_id"].astype(str)
    frames.append(frame)
targets = pd.concat(frames, ignore_index=True)
canonical = targets[["target_id", "RA", "DEC"]].rename(columns={"RA": "ra_deg", "DEC": "dec_deg"})
print(f"Total watchlist targets: {len(targets):,}")
print(targets["grade"].value_counts(dropna=False))


Total watchlist targets: 35,400
grade
D              14824
E               8216
unavailable     5756
C               4692
B               1786
A                126
Name: count, dtype: int64


## 3. Visitas individuales por banda

`query_visit_summary_by_band` consulta DP2 `VisitDetector` por tiles espaciales, recupera las filas detector-visita candidatas y verifica localmente la geometría de los cuatro vértices del detector. Se ejecuta una sola vez para el catálogo `watchlist`, usando la colección seleccionada en `COLLECTION_NAME`. Cada tile queda cacheada en `CACHE_ROOT/visit_tiles`, por lo que una interrupción puede reanudarse.

El resultado agregado tiene una fila por target y banda. También se conserva una tabla detallada con una fila por asociación target–visita–detector.

In [3]:
precomputed_coverage_path = OUTPUT_ROOT / "coverage_and_coadds_all.csv"
expected_ids = set(canonical["target_id"].astype(str))
if precomputed_coverage_path.exists():
    cached_coverage = pd.read_csv(precomputed_coverage_path)
    cached_ids = set(cached_coverage["target_id"].astype(str))
else:
    cached_coverage, cached_ids = None, set()

if cached_coverage is not None and expected_ids.issubset(cached_ids):
    coverage = cached_coverage.loc[cached_coverage["target_id"].astype(str).isin(expected_ids)].copy()
    print(f"Reutilizado sin consultar Butler: {len(coverage):,} targets desde {precomputed_coverage_path}")
else:
    coverage = COLLECTION.get_visit_coverage(canonical, 
                                        tile_nside=TILE_NSIDE, max_workers=TAP_WORKERS, timeout_seconds=7200, 
                                        search_radius_deg=SEARCH_RADIUS_DEG, cache_dir=CACHE_ROOT / "visit_coverage", bands=BANDS, verbose=True)
# Restore catalog metadata when coverage comes from a cache created without it.
metadata_columns = ["target_id", "catalog", "RA", "DEC"]
for optional_column in ("grade", "Status", "DB_JNAME"):
    if optional_column in targets.columns:
        metadata_columns.append(optional_column)
metadata = targets[metadata_columns].drop_duplicates("target_id")
coverage = coverage.drop(columns=[column for column in metadata_columns if column != "target_id" and column in coverage.columns], errors="ignore")
coverage = coverage.merge(metadata, on="target_id", how="left")



Reutilizado sin consultar Butler: 35,400 targets desde /home/karennowo/target_selection/notebooks/outputs/lastberu_dp2/coverage_and_coadds_all.csv


/tmp/ipykernel_3622/1339182677.py:4: DtypeWarning: Columns (90) have mixed types. Specify dtype option on import or set low_memory=False.
  cached_coverage = pd.read_csv(precomputed_coverage_path)


## 4. Coadds y propiedades de calidad

Se comprueba en Butler si existe un coadd real para cada target con cobertura individual. La consulta directa se guarda en cache. Luego se muestrean, también con cache, los mapas consolidados de profundidad, PSF, tiempo de exposición y ruido de cielo. Esto evita repetir estas lecturas en futuras ejecuciones.

In [4]:
if "has_coadd" not in coverage.columns:
    coverage = COLLECTION.get_coadd_coverage(coverage, 
                                       cache_path=CACHE_ROOT / "direct_coadd_coverage.csv", reuse_cache=True, bands=BANDS, verbose=True)
else:
    print("Cobertura de coadds reutilizada desde el resultado completo.")

Cobertura de coadds reutilizada desde el resultado completo.


In [5]:
property_columns = [f"coadd_maglim_{band}" for band in BANDS]
if not set(property_columns).issubset(coverage.columns):
    property_targets = coverage.loc[coverage.has_coadd, ["target_id", "ra_deg", "dec_deg"]].copy()
    property_key = hashlib.sha256("|".join(sorted(property_targets["target_id"].astype(str))).encode()).hexdigest()[:16]
    property_cache = CACHE_ROOT / f"coadd_properties_{property_key}.csv"
    if len(property_targets):
        if property_cache.exists():
            properties = pd.read_csv(property_cache)
            print(f"Reutilizado: {property_cache}")
        else:
            properties = COLLECTION.get_property_map_values(property_targets, properties=("maglim", "psf_size", "exposure_time", "sky_noise"), partial_reads=True, verbose=True)
            properties.to_csv(property_cache, index=False)
        coverage = coverage.merge(properties, on="target_id", how="left", suffixes=("", "_map"))
else:
    print("Propiedades de coadds reutilizadas desde el resultado completo.")
for band in BANDS:
    maglim_values = pd.to_numeric(coverage[f"coadd_maglim_{band}"], errors="coerce") if f"coadd_maglim_{band}" in coverage else pd.Series(np.nan, index=coverage.index)
    psf_values = pd.to_numeric(coverage[f"coadd_psf_fwhm_arcsec_{band}"], errors="coerce") if f"coadd_psf_fwhm_arcsec_{band}" in coverage else pd.Series(np.nan, index=coverage.index)
    metric_mask = maglim_values.notna() & psf_values.notna()
    coverage[f"has_valid_coadd_{band}"] = coverage[f"has_coadd_{band}"].fillna(False).astype(bool) & metric_mask
coverage["has_valid_coadd"] = coverage[[f"has_valid_coadd_{band}" for band in BANDS]].any(axis=1)
coverage["valid_coadd_n_bands"] = coverage[[f"has_valid_coadd_{band}" for band in BANDS]].sum(axis=1).astype(int)
coverage = coverage.loc[:, ~coverage.columns.astype(str).str.startswith("Unnamed:")]
coverage.to_csv(OUTPUT_ROOT / "coverage_and_coadds_all.csv", index=False)
print(f"Targets with coadd coverage: {coverage.has_coadd.sum():,}/{len(coverage):,}")

Propiedades de coadds reutilizadas desde el resultado completo.
Targets with coadd coverage: 4,929/35,400


## 5. Separación de resultados y tablas resumen

A partir de `coverage_and_coadds_all.csv` se generan, sin nuevas consultas, los archivos de `watchlist`. `summary_by_band.csv` responde cobertura e imágenes por filtro; `summary_by_grade.csv` resume la información por categoría `grade`. También se guarda el catálogo completo de cada grupo.

In [6]:
def numeric(frame, column):
    return pd.to_numeric(frame[column], errors="coerce") if column in frame else pd.Series(np.nan, index=frame.index)

def write_summaries(group, name, out):
    group = group.copy()
    group = group.loc[:, ~group.columns.astype(str).str.startswith("Unnamed:")]
    rows = []
    for status in sorted(group["grade"].dropna().unique(), key=str):
        subset = group.loc[group["grade"].eq(status)]
        status_text = str(status).strip()
        display_grade = "unknown" if status_text.lower() in {"", "nan", "none", "unavailable"} else status_text
        rows.append({"grade": display_grade, "type": "visits", "total": int(subset["has_visit_coverage"].fillna(False).astype(bool).sum()), **{band: int(numeric(subset, f"n_images_{band}").fillna(0).gt(0).sum()) for band in BANDS}})
        rows.append({"grade": display_grade, "type": "coadds_found", "total": int(subset["has_coadd"].fillna(False).astype(bool).sum()), **{band: int(subset[f"has_coadd_{band}"].fillna(False).astype(bool).sum()) for band in BANDS}})
        rows.append({"grade": display_grade, "type": "coadds_with_valid_metrics", "total": int(subset["has_valid_coadd"].fillna(False).astype(bool).sum()), **{band: int(subset[f"has_valid_coadd_{band}"].fillna(False).astype(bool).sum()) for band in BANDS}})
    summary = pd.DataFrame(rows, columns=["grade", "type", "total", *BANDS])
    summary.to_csv(out / "summary_by_grade.csv", index=False)
    band_rows = [{"band": band, "targets_with_visits": int(numeric(group, f"n_images_{band}").fillna(0).gt(0).sum()), "coadds_found": int(group[f"has_coadd_{band}"].fillna(False).astype(bool).sum()), "coadds_with_valid_metrics": int(group[f"has_valid_coadd_{band}"].fillna(False).astype(bool).sum())} for band in BANDS]
    band_rows.append({"band": "total", "targets_with_visits": int(group["has_visit_coverage"].fillna(False).astype(bool).sum()), "coadds_found": int(group["has_coadd"].fillna(False).astype(bool).sum()), "coadds_with_valid_metrics": int(group["has_valid_coadd"].fillna(False).astype(bool).sum())})
    band_summary = pd.DataFrame(band_rows)
    band_summary.to_csv(out / "summary_by_band.csv", index=False)
    shown_band = band_summary.rename(columns={"band": "Band", "targets_with_visits": "Targets with visits", "coadds_found": "Coadds found", "coadds_with_valid_metrics": "Coadds with valid metrics"}).copy()
    for column in shown_band.columns[1:]: shown_band[column] = shown_band[column].map(lambda value: f"{int(value):,}")
    fig, ax = plt.subplots(figsize=(7.0, 2.1), dpi=220); ax.axis("off")
    table = ax.table(cellText=shown_band.values, colLabels=list(shown_band.columns), cellLoc="center", loc="center", bbox=[0, 0, 1, .86]); table.auto_set_font_size(False); table.set_fontsize(6)
    for (row, column), cell in table.get_celld().items():
        cell.set_facecolor("white"); cell.set_edgecolor("black"); cell.set_linewidth(0); cell.PAD = .025
        if row == 0: cell.visible_edges = "TB"; cell.set_linewidth(.8); cell.set_text_props(weight="bold")
        elif row == len(shown_band): cell.visible_edges = "B"; cell.set_linewidth(.8)
    ax.set_title(f"LaStBeRu watchlist × DP2 — {name}: coverage by band", fontsize=9, fontweight="bold", y=.975, pad=0); fig.savefig(out / "summary_by_band.png", dpi=220, bbox_inches="tight", pad_inches=.02); plt.close(fig)
    visit_columns = ["target_id", "grade", "RA", "DEC", *[f"n_images_{band}" for band in BANDS], "n_images_total", "has_visit_coverage"]
    group[[column for column in visit_columns if column in group]].to_csv(out / "individual_visits_by_target.csv", index=False)
    group.to_csv(out / "coverage_and_coadds.csv", index=False)
    shown = summary.rename(columns={"grade": "Grade", "type": "Type", "total": "Total"}).copy()
    shown["Type"] = shown["Type"].map({"visits": "Visits", "coadds_found": "Coadds found", "coadds_with_valid_metrics": "Coadds-valid metrics"})
    for column in shown.columns[2:]: shown[column] = shown[column].map(lambda value: f"{int(value):,}")
    fig, ax = plt.subplots(figsize=(8.5, 4.1), dpi=220); ax.axis("off")
    table = ax.table(cellText=shown.values, colLabels=list(shown.columns), cellLoc="center", loc="center", bbox=[0, .015, 1, .91])
    table.auto_set_font_size(False); table.set_fontsize(7.0)
    [cell.get_text().set_fontsize(5) for (row, col), cell in table.get_celld().items() if col == 1 and row > 0]
    for (row, column), cell in table.get_celld().items():
        cell.set_facecolor("white"); cell.set_edgecolor("black"); cell.set_linewidth(0); cell.PAD = .025
        if row == 0: cell.visible_edges = "TB"; cell.set_linewidth(.8); cell.set_text_props(weight="bold")
        elif row == len(shown): cell.visible_edges = "B"; cell.set_linewidth(.8)
        if column == 0 and row > 0: cell.set_text_props(weight="bold")
    ax.set_title(f"LaStBeRu watchlist × DP2 — {name}: visits and coadds", fontsize=9, fontweight="bold", pad=2)
    fig.savefig(out / "summary_by_grade.png", dpi=220, bbox_inches="tight", pad_inches=.02); plt.close(fig)
    return summary

summaries = {}
for name in ("watchlist",):
    folder = OUTPUT_ROOT / name; folder.mkdir(exist_ok=True)
    summaries[name] = write_summaries(coverage.loc[coverage.catalog.eq(name)].copy(), name, folder)
    print(f"{name}: {len(coverage.loc[coverage.catalog.eq(name)]):,} targets; outputs in {folder}")

watchlist: 35,400 targets; outputs in /home/karennowo/target_selection/notebooks/outputs/lastberu_dp2/watchlist


In [7]:
coverage.loc[coverage.catalog.eq(name)][["grade", "n_images_u"]]

,grade,n_images_u
0,unavailable,0
1,E,0
2,E,0
3,E,0
4,D,0
...,...,...
35395,E,0
35396,D,0
35397,unavailable,0
35398,E,0


## 6. Distribuciones

Para cada catálogo se guardan histogramas de visitas individuales, filtros con visitas, filtros con coadd, profundidad promedio y PSF promedio. Los valores se calculan usando el catálogo ya enriquecido.

In [8]:
from matplotlib.ticker import MaxNLocator

for name in ("watchlist",):
    g = coverage.loc[coverage.catalog.eq(name)].copy()
    # Rebuild derived counters when an older coverage cache does not contain them.
    visit_band_columns = [f"n_images_{band}" for band in BANDS if f"n_images_{band}" in g]
    coadd_band_columns = [f"has_coadd_{band}" for band in BANDS if f"has_coadd_{band}" in g]
    valid_coadd_band_columns = [f"has_valid_coadd_{band}" for band in BANDS if f"has_valid_coadd_{band}" in g]
    if "n_images_total" not in g:
        g["n_images_total"] = g[visit_band_columns].fillna(0).sum(axis=1) if visit_band_columns else 0
    if "n_filters_with_images" not in g:
        g["n_filters_with_images"] = g[visit_band_columns].fillna(0).gt(0).sum(axis=1) if visit_band_columns else 0
    if "coadd_n_bands" not in g:
        g["coadd_n_bands"] = g[coadd_band_columns].fillna(False).astype(bool).sum(axis=1) if coadd_band_columns else 0
    if "valid_coadd_n_bands" not in g:
        g["valid_coadd_n_bands"] = g[valid_coadd_band_columns].fillna(False).astype(bool).sum(axis=1) if valid_coadd_band_columns else 0
    def ncol(b): return numeric(g, f"n_images_{b}").fillna(0)
    visit_arrays = [ncol(b) for b in BANDS]
    colors = dict(zip(BANDS, plt.get_cmap("tab10").colors[:len(BANDS)]))
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    max_visits = max(int(x.max()) for x in visit_arrays) if visit_arrays else 1
    axes[0, 0].hist(visit_arrays, bins=np.arange(-.5, max_visits + 1.5, 1), stacked=True, color=[colors[b] for b in BANDS], label=BANDS, edgecolor="none")
    axes[0, 0].set(title="Individual visits per target", xlabel="Number of visits", ylabel="Targets"); axes[0, 0].legend(title="Band", ncol=3, fontsize=8); axes[0, 0].grid(alpha=.25); axes[0, 0].xaxis.set_major_locator(MaxNLocator(nbins=8, integer=True))
    x = np.arange(len(BANDS)); width = .25
    visit_targets = [int(ncol(b).gt(0).sum()) for b in BANDS]
    found_targets = [int(g[f"has_coadd_{b}"].fillna(False).astype(bool).sum()) for b in BANDS]
    valid_targets = [int(g[f"has_valid_coadd_{b}"].fillna(False).astype(bool).sum()) for b in BANDS]
    axes[0, 1].bar(x-width, visit_targets, width, label="Visits", color="tab:blue"); axes[0, 1].bar(x, found_targets, width, label="Coadds found", color="tab:orange"); axes[0, 1].bar(x+width, valid_targets, width, label="Coadds with valid metrics", color="tab:green")
    axes[0, 1].set(xticks=x, xticklabels=BANDS, title="Visits and coadd coverage by band", xlabel="Band", ylabel="Targets"); axes[0, 1].legend(fontsize=8); axes[0, 1].grid(alpha=.25, axis="y")
    n_visit_filters = pd.concat(visit_arrays, axis=1).gt(0).sum(axis=1); n_found_filters = g[[f"has_coadd_{b}" for b in BANDS]].fillna(False).sum(axis=1); n_valid_filters = g[[f"has_valid_coadd_{b}" for b in BANDS]].fillna(False).sum(axis=1)
    axes[0, 2].hist([n_visit_filters, n_found_filters, n_valid_filters], bins=np.arange(-.5, len(BANDS)+1.5, 1), label=["Visits", "Coadds found", "Coadds with valid metrics"], color=["tab:blue", "tab:orange", "tab:green"], alpha=.65, rwidth=.85)
    axes[0, 2].set(xticks=np.arange(len(BANDS)+1), title="Number of bands per target", xlabel="Number of bands", ylabel="Targets"); axes[0, 2].legend(fontsize=8); axes[0, 2].grid(alpha=.25); axes[0, 2].xaxis.set_major_locator(MaxNLocator(integer=True))
    def overlay(ax, columns, title, xlabel):
        vals = [numeric(g, col).dropna() for col in columns]; vals = [v for v in vals if len(v)]
        if vals:
            edges = np.histogram_bin_edges(pd.concat(vals), bins=25)
            for b, col in zip(BANDS, columns):
                v = numeric(g, col).dropna()
                if len(v): ax.hist(v, bins=edges, color=colors[b], edgecolor=colors[b], alpha=.22, linewidth=1.4, label=b)
        ax.set(title=title, xlabel=xlabel, ylabel="Targets"); ax.legend(title="Band", ncol=3, fontsize=8); ax.grid(alpha=.25); ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
    overlay(axes[1, 0], [f"coadd_maglim_{b}" for b in BANDS], "Coadd depth for targets with metrics", "Limiting magnitude")
    overlay(axes[1, 1], [f"coadd_psf_fwhm_arcsec_{b}" for b in BANDS], "Coadd PSF FWHM for targets with metrics", "FWHM [arcsec]")
    overlay(axes[1, 2], [f"visit_seeing_mean_arcsec_{b}" for b in BANDS], "Visit seeing by band", "Seeing [arcsec]")
    fig.suptitle(f"LaStBeRu watchlist × DP2 — {name}", fontsize=13); fig.tight_layout(); fig.savefig(OUTPUT_ROOT / name / "distributions.png", dpi=180); plt.close(fig)

    g["category"] = g["grade"].map(lambda value: f"Grade={value}")
    g["coadd_depth_mean"] = g[[f"coadd_maglim_{b}" for b in BANDS if f"coadd_maglim_{b}" in g]].mean(axis=1)
    g["coadd_psf_mean"] = g[[f"coadd_psf_fwhm_arcsec_{b}" for b in BANDS if f"coadd_psf_fwhm_arcsec_{b}" in g]].mean(axis=1)
    g["visit_seeing_mean"] = g[[f"visit_seeing_mean_arcsec_{b}" for b in BANDS if f"visit_seeing_mean_arcsec_{b}" in g]].mean(axis=1)
    category_colors = dict(zip(sorted(g["category"].unique()), plt.get_cmap("tab10").colors))
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for category, color in category_colors.items():
        h = g.loc[g["category"].eq(category)]
        axes[0, 0].hist(h["n_images_total"], bins=30, histtype="step", linewidth=1.5, alpha=.8, color=color, label=category)
        axes[0, 1].hist(h["n_filters_with_images"], bins=np.arange(-.5, len(BANDS)+1.5), histtype="step", linewidth=1.5, alpha=.8, color=color)
        axes[0, 2].hist(h["coadd_n_bands"], bins=np.arange(-.5, len(BANDS)+1.5), histtype="step", linewidth=1.5, alpha=.8, color=color, label=f"{category} — found")
        axes[0, 2].hist(h["valid_coadd_n_bands"], bins=np.arange(-.5, len(BANDS)+1.5), histtype="step", linewidth=1.5, alpha=.8, color=color, linestyle="--", label=f"{category} — valid metrics")
        for axis, column in zip(axes[1], ("coadd_depth_mean", "coadd_psf_mean", "visit_seeing_mean")):
            values = pd.to_numeric(h[column], errors="coerce").dropna()
            if len(values): axis.hist(values, bins=25, histtype="step", linewidth=1.5, alpha=.8, color=color, label=category)
    for axis, title, xlabel in zip(axes.flat, ("Individual visits", "Filters with visits", "Coadd bands: found vs valid", "Coadd depth", "Coadd PSF FWHM", "Visit seeing"), ("Number of visits", "Number of filters", "Number of bands", "Limiting magnitude", "FWHM [arcsec]", "Seeing [arcsec]")):
        axis.set(title=f"{title} by Grade", xlabel=xlabel, ylabel="Targets"); axis.grid(alpha=.25); axis.xaxis.set_major_locator(MaxNLocator(nbins=8))
    axes[0, 0].legend(title="Grade", fontsize=8); axes[0, 2].legend(fontsize=7)
    fig.suptitle(f"LaStBeRu watchlist × DP2 — {name} — distributions by Grade", fontsize=13); fig.tight_layout(); fig.savefig(OUTPUT_ROOT / name / "distributions_by_grade.png", dpi=180); plt.close(fig)

## 7. Mapas del cielo

Cada punto corresponde a un target. El color representa la cantidad de visitas individuales; se guarda un mapa total y uno por banda para cada catálogo. Se usa una escala logarítmica cuando hay valores positivos para distinguir targets con pocas y muchas visitas.

In [9]:
from matplotlib.colors import LogNorm

def save_map(frame, count_col, path, title):
    values = numeric(frame, count_col).fillna(0)
    fig, ax = plt.subplots(figsize=(10, 5), dpi=180)
    if values.max() > 0:
        sc = ax.scatter(frame.RA % 360, frame.DEC, c=values, s=7, cmap="viridis", norm=LogNorm(vmin=1, vmax=max(1, values.max())), linewidths=0)
    else: sc = ax.scatter(frame.RA % 360, frame.DEC, c=values, s=7, cmap="viridis", linewidths=0)
    ax.invert_xaxis(); ax.grid(alpha=.2); ax.set(xlabel="RA [deg]", ylabel="DEC [deg]", title=title); fig.colorbar(sc, ax=ax, label="Individual visits"); fig.tight_layout(); fig.savefig(path, bbox_inches="tight"); plt.close(fig)

for name in ("watchlist",):
    g = coverage.loc[coverage.catalog.eq(name)]
    save_map(g, "n_images_total", OUTPUT_ROOT / name / "sky_map_visits_total.png", f"{name} — individual total visits")
    for b in BANDS: save_map(g, f"n_images_{b}", OUTPUT_ROOT / name / f"sky_map_visits_{b}.png", f"{name} — individual total visits in {b}")


## 8. Optional cutouts

Cutout generation is disabled by default for the 35,400-target watchlist. Set `MAKE_CUTOUTS=True` to save FITS and PNG cutouts under `/deleted-sundays/nowokaren/lastberu_dp2_cutouts`.


In [10]:
if MAKE_CUTOUTS:
    manifest = COLLECTION.get_coadd_cutouts(
        coverage,
        output_dir=PNG_ROOT, fits_dir=FITS_ROOT,
        bands=BANDS, size_arcsec=CUTOUT_SIZE_ARCSEC, category_column="grade", manifest_path=OUTPUT_ROOT / "watchlist" / "cutout_manifest.csv",
        add_rgb=False, reuse_existing=True, verbose=True,
    )
    manifest.to_csv(OUTPUT_ROOT / "watchlist" / "cutout_manifest.csv", index=False)
    print(f"FITS saved under: {FITS_ROOT}")
    print(f"PNG cutouts saved under: {PNG_ROOT}")
else:
    print("MAKE_CUTOUTS=False: no cutouts generated.")


Cutouts DP2:   0%|          | 0/4929 [00:00<?, ?it/s]

FITS saved under: /deleted-sundays/nowokaren/lastberu_dp2_cutouts/fits
PNG cutouts saved under: /deleted-sundays/nowokaren/lastberu_dp2_cutouts/pngs


In [11]:
FITS_ROOT

PosixPath('/deleted-sundays/nowokaren/lastberu_dp2_cutouts/fits')

## 10. Archivos finales

Al finalizar, los productos quedan organizados así:

```text
outputs/lastberu_dp2/
├── │   ├── coverage_and_coadds.csv
│   ├── summary_by_band.csv
│   ├── summary_by_grade.csv
│   ├── distributions.png
│   ├── sky_map_visits_*.png
│   ├── cutout_manifest.csv
│   ├── cutout_manifest.csv
│   └── (cutouts in delete_sundays/lastberu_cutouts/<catalog>/)
├── watchlist/
│   └── (misma estructura)
└── cache/

delete_sundays/lastberu_cutouts/
├── │   ├── fits/
│   ├── cutouts/
│   └── rgb/
└── watchlist/
    ├── fits/
    ├── cutouts/
```

The PDF is generated by the report cell in section 11. La consulta de cobertura puede reutilizarse dejando `reuse_cache=True`; para forzar una actualización habría que borrar o cambiar la cache correspondiente.

## 11. Output products

The output directory contains English PNG figures and CSV tables for the full watchlist.


## Refresh summaries after cutout generation

This cell refreshes the summary tables after the cutout manifest has been written, including coadds with no valid pixels.


In [12]:
from target_selection.reporting import write_collection_summaries

summary_coverage = pd.read_csv(OUTPUT_ROOT / "watchlist" / "coverage_and_coadds.csv", low_memory=False)
summary_manifest = OUTPUT_ROOT / "watchlist" / "cutout_manifest.csv"
summary_group = summary_coverage.loc[summary_coverage["catalog"].eq("watchlist")].copy() if "catalog" in summary_coverage else summary_coverage
write_collection_summaries(summary_group, OUTPUT_ROOT / "watchlist", bands=BANDS, category_column="grade", category_label="Grade", name="LaStBeRu watchlist × DP2", cutout_manifest=summary_manifest)
print("Summary tables and figures refreshed from the cutout manifest.")


Summary tables and figures refreshed from the cutout manifest.


In [ ]:
pdf_path = build_collection_report_pdf(
    coverage, fits_dir=FITS_ROOT, asset_dir=OUTPUT_ROOT / "watchlist",
    output_path=OUTPUT_ROOT / "watchlist" / "lastberu_dp2_report.pdf",
    catalog_name="watchlist", bands=BANDS, manifest_path=OUTPUT_ROOT / "watchlist" / "cutout_manifest.csv", verbose=True,
)
print(f"Report PDF saved: {pdf_path}")


Report target pages: 4,929 coadd targets; 13 currently without verified FITS.


Report target pages:   0%|          | 0/4929 [00:00<?, ?it/s]

In [ ]:
print("Main outputs — watchlist:")
for filename in ("coverage_and_coadds.csv", "summary_by_band.csv", "summary_by_grade.csv", "distributions.png", "distributions_by_grade.png", "individual_visits_by_target.csv"):
    print(OUTPUT_ROOT / "watchlist" / filename)


In [ ]:
watchlist = pd.read_csv(
    "outputs/lastberu_dp2/watchlist/coverage_and_coadds.csv"
)

print("Watchlist grade summary:")
display(watchlist.groupby("grade").agg(
    targets=("target_id", "size"),
    targets_with_visits=("n_images_total", lambda x: pd.to_numeric(x, errors="coerce").fillna(0).gt(0).sum()),
    targets_with_coadd=("has_coadd", "sum"),
))